In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import numpy as np


Instructions for updating:
non-resource variables are not supported in the long term


# 딥러닝(deep learning)

인간의 뇌는 뉴런이라는 세포로 이루어져 있고, 약 1,000억 개의 뉴런이 존재한다. 그리고 각 뉴런은 시냅스로 연결되는데, 자극의 크기가 특정값 이상이면 다음 뉴런으로 전달되고, 특정값 이하라면 다음 뉴런에 전달되지 않는다. 이때, 자극을 전달하는 기준인 특정값을 임계치라고 한다. 그래서 우리는 임계치를 넘긴 자극 신호를 전달받고 그에 해당하는 반응을 하게 되는 것이다.

이와 같은 생명 과학 분야의 아이디어를 머신러닝에 도입한 것을 `인공 신경 망(Artifical Neural Network, ANN)`이라고 하며 줄이면 신경망이라고 한다. 그리고 `신경망을 기반으로 하는 학습하는 방법을 딥러닝`이라고 부른다. 수많은 뉴런이 서로 연결되어 신호를 전달하는 것 처럼 딥러닝에서는 수많은 퍼셉트론이 연결되어 있어 연산 결과를 주고 받는다.

AND 연산이나 OR 연산의 경우 적합한 직선 하나를 생성해서 출력값 0과 1을 정확히 구분해 낼 수 있었다. 그러나 XOR 연산은 직선 하나로는 0과 1을 구분할 수 없다. 초기 딥러닝은 `XOR 문제를 해결하는 방법으로 등장`했고 신경망을 이용해서 기존 방법으로는 풀 수 없었던 XOR 문제를 풀 수 있었다. XOR 문제와 같은 간단한 문제를 해결하는 것으로 시작한 딥러닝은 점차 발전해 더욱 복잡한 문제를 푸는데 사용되었다.

<img src="./deepLearning1.png" width="1400" align="left" />

<img src="./deepLearning2.png" width="1400" align="left" />

<img src="./perceptron1.png" width="1400" align="left" />

<img src="./perceptron2.png" width="1400" align="left" />

<img src="./perceptron3.png" width="1400" align="left" />

퍼셉트론은 신경망의 최소 단위로 입력값과 출력값이 존재한다. n개의 입력을 합쳐서 벡터 $x$로 표현하고, n개의 가중치 값을 가중치 벡터 $w$라고 한다. 그리고 입력값 벡터와 가중치 벡터의 내적값은 출력값 $z$가 된다.

<img src="./perceptron.png" width="600" align="left" />

# 퍼셉트론 AND, OR, XOR 연산

퍼셉트론은 엄밀하게 말하면 딥러닝은 아니지만 딥러닝과 상당히 닮은 분류기이다.  
뉴런의 계산 과정을 이해하고 뉴런 하나만으로 AND 연산 또는 OR 연산과 같은 기초적인 연산을 할 수 있음을 알아본다.

## 퍼셉트론 코드의 이해도를 높이기 위해 1을 T로 0을 F로 그리고 편향은 1로 지정한다.

In [2]:
T = 1.0
F = 0.0
B = 1.0

## AND, OR, XOR 연산의 실습 데이터를 함수로 정의한다.

In [3]:
def and_data():
    # 학습 데이터
    X = [
        [F, F, B], # F
        [F, T, B], # F
        [T, F, B], # F
        [T, T, B], # T
    ]
    # 학습 데이터에 따른 레이블(실제값)
    Y = [
        [F],
        [F],
        [F],
        [T]
    ]
    return X, Y

def or_data():
    # 학습 데이터
    X = [
        [F, F, B], # F
        [F, T, B], # T
        [T, F, B], # T
        [T, T, B], # T
    ]
    # 학습 데이터에 따른 레이블(실제값)
    Y = [
        [F],
        [T],
        [T],
        [T]
    ]
    return X, Y

def xor_data():
    # 학습 데이터
    X = [
        [F, F, B], # F
        [F, T, B], # T
        [T, F, B], # T
        [T, T, B], # F
    ]
    # 학습 데이터에 따른 레이블(실제값)
    Y = [
        [F],
        [T],
        [T],
        [F]
    ]
    return X, Y

In [4]:
# X, Y = and_data() # and 연산에 사용할 데이터를 호출한다.
# X, Y = or_data() # or 연산에 사용할 데이터를 호출한다.
X, Y = xor_data() # xor 연산에 사용할 데이터를 호출한다.
print(X, Y)

[[0.0, 0.0, 1.0], [0.0, 1.0, 1.0], [1.0, 0.0, 1.0], [1.0, 1.0, 1.0]] [[0.0], [1.0], [1.0], [0.0]]


# tenserflow를 사용해서 퍼셉트론을 구현한다.

## 가중치와 바이어스를 랜덤하게 설정한다.

In [5]:
# 3행 1열인 균등 분포를 따르는 난수를 만든다.
W = tf.Variable(tf.random_uniform([3, 1], dtype=tf.float32)) # 0행 0열과 1행 0열은 가중치로 사용하고 2행 0열을 바이어스로 사용한다.

sess = tf.Session()
sess.run(tf.global_variables_initializer())
print(sess.run(W))

[[0.79491365]
 [0.12166929]
 [0.5942557 ]]


## 활성화(step) 함수를 만든다.

퍼셉트론은 두 개의 입력이 있을 때 하나의 뉴런으로 두 개의 입력을 계산한 뒤 최종적으로 0 또는 1을 출력한다.

두 개의 입력은 가중치와 곱해져서 뉴런 첫 번째 단계인 시그마로 들어가고 시그마 단계에서 입력값에 가중치를 곱한 결과에 바이어스를 더하는 과정이 이루어진다.  
시그마에서 연산된 결과는 뉴런의 두 번째 단계인 활성화 함수의 입력으로 들어가는데 퍼셉트론은 스텝 함수라는 활성화 함수를 사용하고 스텝 함수는 시그마 값이 0 이하일 경우 0을, 0 초과일 경우 1을 리턴한다.

In [6]:
print(sess.run(tf.greater(1, 0))) # 1 > 0
print(sess.run(tf.greater_equal(1, 0))) # 1 >= 0
print(sess.run(tf.less(1, 0))) # 1 < 0
print(sess.run(tf.less_equal(1, 0))) # 1 <= 0
print(sess.run(tf.equal(1, 0))) # 1 == 0
print(sess.run(tf.not_equal(1, 0))) # 1 != 0

True
True
False
False
False
True


In [7]:
print(sess.run(tf.greater(1, 0)))
print(sess.run(tf.cast(tf.greater(1, 0), dtype=tf.float32)))
print(sess.run(tf.greater(1, 1)))
print(sess.run(tf.cast(tf.greater(1, 1), dtype=tf.float32)))

True
1.0
False
0.0


인수로 넘겨받은 입력값에 가중치를 곱한 결과에 바이어스를 더한값(x)과 0과 비교해서 x가 0보다 크면 1.0을 그렇치 않으면 0.0를 리턴하는 함수

In [8]:
def step(x):
    return tf.cast(tf.greater(x, 0), dtype=tf.float32)

In [9]:
print(sess.run(step(0.0001)))
print(sess.run(step(0.0)))

1.0
0.0


## 시그마(퍼셉트론의 가설)를 만든다.

In [10]:
Z = tf.matmul(X, W)
print('예측값\n', sess.run(Z), sep='')

예측값
[[0.5942557]
 [0.715925 ]
 [1.3891693]
 [1.5108386]]


## 시그마를 활성화 함수에 넣어서 퍼셉트론의 결과를 얻는다.

In [11]:
result = step(Z)
print('최종 예측값\n', sess.run(result), sep='')

최종 예측값
[[1.]
 [1.]
 [1.]
 [1.]]


## 실제값과 최종 예측값의 오차를 계산한다.

In [12]:
loss = tf.subtract(Y, result)
print('실제값과 최종 예측값의 오차\n', sess.run(loss), sep='')

실제값과 최종 예측값의 오차
[[-1.]
 [ 0.]
 [ 0.]
 [-1.]]


## 평균 제곱 오차(MSE)를 계산한다.

In [13]:
mse = tf.reduce_mean(tf.square(loss))
print('평균 제곱 오차\n', sess.run(mse), sep='')

평균 제곱 오차
0.5


# 평균 제곱 오차를 최소가 되게 가중치와 바이어스를 최적화한다.

보통의 경우 딥러닝 모델은 경사 하강법을 통해 매개 변수를 최적화하지만 미분값이 큰 영향이 없는 스텝 함수의 영향으로(0은 미분 불가능, 1(상수)는 미분하면 1) 퍼셉트론은 아래와 같은 방법으로 최적화를 진행한다.

## 퍼셉트론에서 가중치와 바이어스를 수정할 값을 계산한다.

결과 값이 1이 나와야 하는데 스텝 함수의 출력이 0인 경우: new_w = old_w + dalta  
결과 값이 0이 나와야 하는데 스텝 함수의 출력이 1인 경우: new_w = old_w - dalta  

In [14]:
print(X)
print(type(np.array(X)))
print(np.array(X))
print(np.array(X).T) # 전치

[[0.0, 0.0, 1.0], [0.0, 1.0, 1.0], [1.0, 0.0, 1.0], [1.0, 1.0, 1.0]]
<class 'numpy.ndarray'>
[[0. 0. 1.]
 [0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 1.]]
[[0. 0. 1. 1.]
 [0. 1. 0. 1.]
 [1. 1. 1. 1.]]


mulmat(A, B, transpose_a=True)는 내적하기 전에 행렬 A를 전치하고 내적한다.  
mulmat(A, B, transpose_b=True)는 내적하기 전에 행렬 B를 전치하고 내적한다.

In [15]:
# matmul() 메소드로 내적하기 전에 X를 먼저 전치하고 내적한다.
delta = tf.matmul(X, loss, transpose_a=True)
print('가중치와 바이어스를 수정할 값\n', sess.run(delta), sep='')

가중치와 바이어스를 수정할 값
[[-1.]
 [-1.]
 [-2.]]


## 가중치와 바이어스를 수정해서 최적화한다. => 역전파

assign() 메소드는 tensorflow 세션 내부에서 변수에 새로운 값을 할당한다. 변수에 저장된 값을 수정한다.

In [16]:
print('수정된 가중치와 바이어스\n', sess.run(tf.add(W, delta)), sep='')
train = tf.assign(W, tf.add(W, delta))

수정된 가중치와 바이어스
[[-0.20508635]
 [-0.8783307 ]
 [-1.4057443 ]]


# 퍼셉트론을 학습하고 테스트한다.

In [17]:
sess = tf.Session()
sess.run(tf.global_variables_initializer())

error, epoch, max_epoch = 1.0, 0, 20 # 오차, 현재 학습 횟수, 최대 학습 횟수
# 오차가 발생되도 현재 학습 횟수가 최대 학습 횟수를 넘어가지 않는 동안 반복한다.
while error > 0.0 and epoch < max_epoch:
    epoch += 1 # 현재 학습 횟수를 1증가시킨다.
    error = sess.run(mse) # 평균 제곱 오차를 계산한다.
    sess.run(train) # 학습시킨다.
    print(f'epoch: {epoch:2d}, mse: {error:.2f}')
    
print('학습 결과\n', sess.run(result), sep='')

epoch:  1, mse: 0.50
epoch:  2, mse: 0.50
epoch:  3, mse: 0.50
epoch:  4, mse: 0.50
epoch:  5, mse: 0.50
epoch:  6, mse: 0.50
epoch:  7, mse: 0.50
epoch:  8, mse: 0.50
epoch:  9, mse: 0.50
epoch: 10, mse: 0.50
epoch: 11, mse: 0.50
epoch: 12, mse: 0.50
epoch: 13, mse: 0.50
epoch: 14, mse: 0.50
epoch: 15, mse: 0.50
epoch: 16, mse: 0.50
epoch: 17, mse: 0.50
epoch: 18, mse: 0.50
epoch: 19, mse: 0.50
epoch: 20, mse: 0.50
학습 결과
[[1.]
 [1.]
 [1.]
 [1.]]
